In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import torch.optim as optim
import matplotlib.pyplot as plt


In [2]:
df=pd.read_csv(r'C:\Coding\ML_DL\Datasets\100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
def tokenize(text):
    text=text.lower()
    text=text.replace('?','')
    text=text.replace("'","")
    return text.split()


In [4]:
#vocab
vocab={'<UNK>':0}

def make_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    merged_tokens=tokenized_question+tokenized_answer

    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)
            


In [5]:
df.apply(make_vocab,axis=1)
(vocab)

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [6]:
def text2indices(text,vocab):
    indexed_text=[]
    for token in tokenize(text):
        if(token in vocab):
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text


In [7]:
class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df=df
        self.vocab=vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self,idx):
        q=text2indices(self.df.iloc[idx]['question'],self.vocab)
        a=text2indices(self.df.iloc[idx]['answer'],self.vocab)

        return torch.tensor(q),torch.tensor(a)

In [8]:
dataset=QADataset(df,vocab)

data_loader=DataLoader(dataset,batch_size=1,shuffle=True)
     

In [61]:
class RNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, X):
        x = self.embedding(X)
        _, h_n = self.rnn(x)
        logits = self.fc(h_n[-1])
        return logits


In [66]:
learning_rate=0.001
epochs=100

In [67]:
model=RNN(len(vocab))

In [68]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [69]:
for epoch in range(epochs):
    total_loss=0
    for question,answer in data_loader:
        # print("question:", question.shape)
        # print("answer:", answer.shape, answer.dtype)
        # break

        optimizer.zero_grad()
        pred=model(question)
        answer = answer.squeeze(1).long()   # (B, 1) → (B,)
        loss=criterion(pred,answer)
        loss.backward()
        optimizer.step()
         
        total_loss+=loss.item()
    
    print(f"Epoch: {epoch+1} Loss:{total_loss}")

Epoch: 1 Loss:527.0113463401794
Epoch: 2 Loss:456.8512177467346
Epoch: 3 Loss:377.89285349845886
Epoch: 4 Loss:313.0933132171631
Epoch: 5 Loss:260.73236107826233
Epoch: 6 Loss:212.8841608762741
Epoch: 7 Loss:169.85070264339447
Epoch: 8 Loss:131.95252841711044
Epoch: 9 Loss:101.3231697678566
Epoch: 10 Loss:77.31757721304893
Epoch: 11 Loss:59.27459305524826
Epoch: 12 Loss:46.40707366168499
Epoch: 13 Loss:36.871689170598984
Epoch: 14 Loss:30.270836248993874
Epoch: 15 Loss:24.854771703481674
Epoch: 16 Loss:20.614187210798264
Epoch: 17 Loss:17.370229862630367
Epoch: 18 Loss:14.814418092370033
Epoch: 19 Loss:12.855505809187889
Epoch: 20 Loss:11.10941220074892
Epoch: 21 Loss:9.76933016255498
Epoch: 22 Loss:8.579720754176378
Epoch: 23 Loss:7.594203926622868
Epoch: 24 Loss:6.812881018966436
Epoch: 25 Loss:6.111738629639149
Epoch: 26 Loss:5.511935293674469
Epoch: 27 Loss:4.977107834070921
Epoch: 28 Loss:4.516931029036641
Epoch: 29 Loss:4.149816047400236
Epoch: 30 Loss:3.795415820553899
Epoch: 31

In [70]:
import torch.nn.functional as F

def predict(model, question, vocab):
    model.eval()

    # text → indices
    num_question = text2indices(question, vocab)

    # tensor + batch dimension
    x = torch.tensor(num_question, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)                    # (1, vocab_size)
        probs = F.softmax(logits, dim=1)     # (1, vocab_size)

        pred_id = probs.argmax(dim=1).item()
        confidence = probs.max(dim=1).values.item()

    return pred_id, confidence


In [73]:
predict(model,"What is the capital of Germany?",vocab)

(9, 0.9991777539253235)

In [75]:
list(vocab.keys())[9]

'berlin'